# Seleção de features (Fase 3)

**Objetivo:** decidir quais colunas da camada Gold `features_aluno` entram no modelo
supervisionado, como cada uma deve ser tratada no pré-processamento, e registrar
o motivo de cada decisão.

**Entrada:** `data/sample/features_aluno` (1 linha por aluno avaliado em 2024).
**Saída:** a lista final de features, usada pelo notebook `03_modelagem_supervisionada`.


## 1. Bibliotecas e configuracões

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 50)

In [2]:
CAMINHO_GOLD = "../data/sample/features_aluno"

df = pd.read_parquet(CAMINHO_GOLD)
df.shape

(1610754, 35)

## 2. Raio-x das colunas

Tipo, número de valores distintos e percentual de nulos de cada coluna.
É a base para separar alvo, identificadores, constantes e features.

In [3]:
raio_x = pd.DataFrame({
    "tipo": df.dtypes,
    "valores_distintos": df.nunique(),
    "pct_nulos": (df.isna().mean() * 100).round(3),
}).sort_values("valores_distintos")

raio_x

,tipo,valores_distintos,pct_nulos
ano,int32,1,0.000
rede_codigo,str,1,0.000
uf_anomala,bool,2,0.000
alfabetizado,bool,2,0.000
mun_elegivel_meta,object,2,0.001
regiao,str,5,0.000
caderno,str,22,0.000
sigla_uf,str,25,0.000
mun_idade_mediana,float64,35,0.000
mun_total_escolas,int32,134,0.000


In [8]:
## Saber qual o valor está nas colunas de ano e rede_codig:

df[["ano", "rede_codigo"]].drop_duplicates()


,ano,rede_codigo
0,2024,3


**Conclusões do raio-x:**

- **Variáveis Constantes:** `ano` (sempre 2024) e `rede_codigo` (sempre "3") têm um único valor
  em todas as linhas, portanto não carregam informação para o modelo e serão excluídas.

- **Variáveis Binárias:** `alfabetizado` é uma variável binária e é justamente o **alvo** do modelo. `uf_anomala` e `mun_elegivel_meta` são candidatas a feature. Um ponto importante é sobr a variável `mun_elegivel_meta` que tem tipo `object` por causa de 0,001% de nulos e precisará de imputação.

- **Variáveis Categóricas de baixa cardinalidade:** `regiao` (5), `sigla_uf` (25) e `caderno` (22)
  são texto e precisarão de encoding.
 
- **Identificadores:** `id_aluno` tem um valor por linha (chave única); `id_escola` (36.920)
  e `id_municipio` (5.450) identificam grupos. Nenhum entra como feature. 
  O `id_municipio` será usado para separar treino e teste.

- **Colunas municipais:** `mun_pib_per_capita` tem exatamente 5.450 valores, um por município.

- **Nulos:** `mun_distancia_meta_anterior` e `mun_meta_ano_alvo` (2,6%, mesmo percentual,
  provavelmente as mesmas linhas), `mun_taxa_ano_anterior` (0,5%), as quatro
  `mun_pct_va_*` (0,006%) e `mun_elegivel_meta` (0,001%). 
  Com todos abaixo de 3% as colunas serão mantidas e os vazios serão tratados por imputação.


## 3. O alvo: `alfabetizado`

**Definição:** o aluno é considerado alfabetizado quando sua proficiência no Saeb é
maior ou igual a 743 pontos (ponto de corte do Indicador Criança Alfabetizada).
Fonte: README do projeto, seção 3 (a regra foi verificada na Silver da Fase 2).

In [9]:
contagem = df["alfabetizado"].value_counts()
percentual = df["alfabetizado"].value_counts(normalize=True).mul(100).round(1)

pd.DataFrame({"alunos": contagem, "percentual": percentual})

,alunos,percentual
alfabetizado,,
True,956343,59.4
False,654411,40.6


**Decisão: inverter o alvo**

No scikit-learn, precisão, recall e F1 são calculados em relação à **classe positiva**,
que por padrão é a classe 1. Na base, o 1 (`True`) é o aluno **alfabetizado**. Treinado
assim, o recall responderia "de todas as crianças alfabetizadas, quantas o modelo
encontrou?", e não é essa a pergunta da secretaria de educação.

O objetivo é **encontrar as crianças em risco**. O erro mais grave é o falso negativo:
uma criança não alfabetizada que o modelo classifica como alfabetizada e que, por isso,
fica fora da ação da política pública.

Por isso o alvo é invertido: **1 = não alfabetizado (risco)** e **0 = alfabetizado**.
A acurácia não é afetada pela inversão; precisão, recall e F1 passam a medir a
capacidade do modelo de encontrar a classe de risco.

In [21]:
# alvo invertido: 1 = não alfabetizado (classe de risco), 0 = alfabetizado
y = (~df["alfabetizado"]).astype(int).rename("nao_alfabetizado")

# prova da inversão: cruza o alvo original com o novo
pd.crosstab(df["alfabetizado"], y, margins=True)

nao_alfabetizado,0,1,All
alfabetizado,,,
False,0,654411,654411
True,956343,0,956343
All,956343,654411,1610754


In [22]:
rotulos = {0: "alfabetizado", 1: "não alfabetizado (risco)"}
y.value_counts().rename(index=rotulos)

nao_alfabetizado
alfabetizado                956343
não alfabetizado (risco)    654411
Name: count, dtype: int64

**Conclusões sobre o alvo:**

- **Balanceamento:** 59,4% dos alunos são alfabetizados e 40,6% não alfabetizados.
  A base é relativamente balanceada, portanto não são necessárias técnicas de balanceamento
  artificial, mas a divisão entre treino e teste deve preservar essa proporção.
- **Baseline ingênuo:** um modelo que responde sempre "alfabetizado" acerta 59,4%
  (acurácia), mas não identifica nenhuma criança em risco (recall de 0% na classe 1).
  O modelo final precisa superar esse patamar, e a acurácia sozinha não basta para
  avaliá-lo.
- **Métrica prioritária:** o erro mais grave é o falso negativo, a criança em risco
  que o modelo classifica como alfabetizada e que fica fora da política pública.
  Por isso a métrica prioritária é o **recall da classe de risco**, acompanhado da
  **precisão** e do **F1**: um modelo que apontasse risco para todos teria recall de
  100% e nenhuma utilidade prática.